# 13 - Full OpenSSH and Apache datasets

Parse the full OpenSSH and Apache logs with Drain3 and cut them into windows: 1-minute windows for OpenSSH, 20-line windows for Apache. Neither dataset has labels, so there is no `Label` column.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features.time_windows import APACHE_HEADERLESS, build_windows
from src.parsing.drain_parser import DATASET_SIM_TH, DEFAULT_SIM_TH

pd.set_option("display.max_colwidth", 120)
PROCESSED = ROOT / "data" / "processed"

## Parse and build windows

Drain depth 4. OpenSSH uses its own masks and similarity threshold 0.7; Apache uses the shared masks and 0.4.

- OpenSSH: fixed 1-minute windows with no overlap; minutes with no lines are dropped.
- Apache: fixed windows of 20 lines in file order, no overlap, last partial window dropped (same as BGL). 1-minute windows were mostly single lines (median 1), so Apache now uses line windows.

In [2]:
results = {}
for dataset in ["OpenSSH", "Apache"]:
    sequences, counts, templates, stats = build_windows(dataset)
    results[dataset] = (sequences, counts, templates, stats)
    print(f"{dataset}: similarity threshold {DATASET_SIM_TH.get(dataset, DEFAULT_SIM_TH)}, {stats['total_seconds']:.1f} s")

Parsing SSH.log: 0 lines [00:00, ? lines/s]

Parsing SSH.log: 655147 lines [00:16, 39836.11 lines/s]

OpenSSH: similarity threshold 0.7, 16.7 s


Parsing Apache.log: 0 lines [00:00, ? lines/s]

Parsing Apache.log: 56482 lines [00:00, 76796.75 lines/s]

Apache: similarity threshold 0.4, 0.8 s


In [3]:
summary = pd.DataFrame({
    dataset: {
        "window": s["window"],
        "lines": f"{s['lines']:,}",
        "unparsed lines": f"{s['unparsed_lines']:,}",
        "headerless lines kept": f"{s['headerless_lines_kept']:,}",
        "templates": s["templates"],
        "windows": f"{s['windows']:,}",
        "dropped (empty minutes / tail lines)": f"{s['dropped']:,}",
        "mean lines per window": f"{s['mean_lines_per_window']:.1f}",
        "median lines per window": f"{s['median_lines_per_window']:.0f}",
        "max lines per window": f"{s['max_lines_per_window']:,}",
        "out-of-order lines": f"{s['out_of_order_lines']:,}",
        "first line": f"{s['first_time']:%Y-%m-%d %H:%M:%S}",
        "last line": f"{s['last_time']:%Y-%m-%d %H:%M:%S}",
    }
    for dataset, (_, _, _, s) in results.items()
})
summary

,OpenSSH,Apache
window,60 s,20 lines
lines,"655,147","56,482"
unparsed lines,0,0
headerless lines kept,0,"4,478"
templates,51,30
windows,"13,486","2,824"
dropped (empty minutes / tail lines),"27,462",2
mean lines per window,48.6,20.0
median lines per window,43,20
max lines per window,341,20


The OpenSSH log has no year, so its dates use a placeholder start year (`OPENSSH_START_YEAR`, 2000) plus one at the Dec to Jan rollover. Only the times between lines matter for the windows.

## Check: headerless Apache lines

Some Apache lines are only the text `script not found or unable to stat`, with no timestamp or level. They are kept and given the timestamp of the line before them.

In [4]:
_, apache_counts, apache_templates, apache_stats = results["Apache"]
event = apache_templates.loc[apache_templates["template"] == APACHE_HEADERLESS, "EventId"].iloc[0]
print(f"Headerless lines kept: {apache_stats['headerless_lines_kept']:,}   unparsed lines: {apache_stats['unparsed_lines']:,}")
print(f"Their event: {event} ({APACHE_HEADERLESS!r}), count over all windows: {apache_counts[event].sum():,}")

Headerless lines kept: 4,478   unparsed lines: 0
Their event: E24 ('script not found or unable to stat'), count over all windows: 4,478


## Check: out-of-order lines

OpenSSH lines go into the minute of their own timestamp, so lines written slightly out of order still go to the right window. Apache windows follow file order, so out-of-order lines stay where they are in the file.

In [5]:
for dataset, (_, _, _, s) in results.items():
    print(f"{dataset}: {s['out_of_order_lines']:,} lines earlier than a previous line")

OpenSSH: 8 lines earlier than a previous line
Apache: 289 lines earlier than a previous line


## Example windows

In [6]:
for dataset, (sequences, _, _, _) in results.items():
    print(dataset)
    for row in sequences.head(3).itertuples():
        start = pd.to_datetime(row.start_time, unit="s")
        seq = " ".join(row.EventSequence)
        print(f"  window {row.window} ({start:%m-%d %H:%M}, {len(row.EventSequence)} lines): {seq[:150]}{' ...' if len(seq) > 150 else ''}")

OpenSSH
  window 0 (12-10 06:55, 7 lines): E1 E2 E3 E4 E5 E6 E7
  window 1 (12-10 07:02, 1 lines): E7
  window 2 (12-10 07:07, 6 lines): E2 E3 E4 E5 E6 E8
Apache
  window 0 (06-09 06:07, 20 lines): E1 E2 E3 E4 E5 E1 E2 E6 E7 E6 E7 E6 E7 E6 E7 E8 E9 E10 E11 E11
  window 1 (06-09 06:07, 20 lines): E11 E11 E11 E11 E11 E11 E12 E13 E12 E12 E12 E12 E12 E12 E12 E14 E14 E14 E14 E14
  window 2 (06-09 14:22, 20 lines): E14 E14 E14 E14 E14 E14 E14 E15 E15 E15 E15 E15 E15 E15 E15 E15 E15 E15 E15 E15


## Templates

The 10 most frequent templates for each dataset.

In [7]:
for dataset, (_, counts, templates, _) in results.items():
    freq = counts.filter(regex=r"^E\d+$").sum().nlargest(10)
    display(pd.DataFrame({"count": freq, "template": freq.index.map(templates.set_index("EventId")["template"])})
            .style.set_caption(dataset))

,count,template
E10,140768,Failed password for from port ssh2
E9,140701,pam_unix(sshd:auth): authentication failure; logname= uid= euid= tty=ssh ruser= <*> user=
E7,68958,Connection closed by [preauth]
E8,46608,Received disconnect from : : <*> <*> [preauth]
E14,37963,PAM service(sshd) ignoring max retries; >
E12,37298,Disconnecting: Too many authentication failures for <*> <*>
E13,37029,PAM more authentication <*> logname= uid= euid= tty=ssh ruser= <*> user=
E11,36967,message repeated times: [ Failed password for from port ssh2]
E6,20271,Failed <*> for invalid user <*> from port ssh2
E4,19852,pam_unix(sshd:auth): check pass; user unknown


,count,template
E15,20861,[client ] File does not exist: <*>
E14,6743,[client ] Directory index forbidden by rule: /var/www/html/
E11,6335,jk2_init() Found child in scoreboard slot
E12,5708,workerEnv.init() ok /etc/httpd/conf/workers2.properties
E24,4478,script not found or unable to stat
E22,4349,mod_jk child workerEnv in error state
E18,3301,[client ] script not found or unable to stat: <*>
E19,1336,mod_jk2 Shutting down
E13,1259,mod_jk child init
E16,971,jk2_init() Can't find child in scoreboard


## Saved files

In [8]:
for name in ["openssh", "apache"]:
    for suffix in ["_sequences.parquet", "_counts.parquet", "_templates.csv"]:
        path = PROCESSED / f"{name}{suffix}"
        print(f"{path.relative_to(ROOT)}  ({path.stat().st_size / 1e6:.1f} MB)")

data/processed/openssh_sequences.parquet  (0.3 MB)
data/processed/openssh_counts.parquet  (0.3 MB)
data/processed/openssh_templates.csv  (0.0 MB)
data/processed/apache_sequences.parquet  (0.1 MB)
data/processed/apache_counts.parquet  (0.1 MB)
data/processed/apache_templates.csv  (0.0 MB)
